In [ ]:
!pip install openai anthropic
!pip install pytest tabulate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.0/355.0 kB 8.2 MB/s eta 0:00:00


In [ ]:
!pip install openai google-generativeai tabulate pytest

In [ ]:
!git clone https://github.com/openai/human-eval.git
%cd human-eval

Cloning into 'human-eval'...
remote: Enumerating objects: 34, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 34 (delta 11), reused 4 (delta 4), pack-reused 9 (from 2)
Receiving objects: 100% (34/34), 55.87 KiB | 866.00 KiB/s, done.
Resolving deltas: 100% (12/12), done.
/content/human-eval


In [ ]:
!pip install evalplus

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 635.4/635.4 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.1/108.1 kB 9.2 MB/s eta 0:00:00
  Created wheel for stop-sequencer: filename=stop_sequencer-1.2.3-py3-none-any.whl size=4869 sha256=e02716d0c834dc2178c0bf880df7b8c4435d91110c97cd9dcfb5fa94324bb0ce
  Stored in directory: /root/.cache/pip/wheels/b4/a3/12/0c6a6541ed0eafd69c075930baea3a103ae49c7d03de988290
  Created wheel for tempdir: filename=tempdir-0.7.1-py3-none-any.whl size=2194 sha256=e011b40a1989b661e3a37a03aeb298ab010c75782e8e4259b91eecf857915745
  Stored in directory: /root/.cache/pip/wheels/1e/14/6b/dedd96c73357be1d37d03b56a8bcfc5425df347e05115ac18c
  Created wheel for wget: fi

In [ ]:
from evalplus.data import get_human_eval_plus

problems = get_human_eval_plus()   # instead of get_human_eval()
print(f"Loaded {len(problems)} problems")

Loaded 164 problems


In [ ]:
first_key = list(problems.keys())[0]
print(first_key)
print(problems[first_key]["prompt"])

HumanEval/0
from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    """



In [ ]:
# selected = list(problems.keys())[:10]
# print("Selected problems:", selected)

Selected problems: ['HumanEval/0', 'HumanEval/1', 'HumanEval/2', 'HumanEval/3', 'HumanEval/4', 'HumanEval/5', 'HumanEval/6', 'HumanEval/7', 'HumanEval/8', 'HumanEval/9']


In [ ]:
import os
os.environ["OPENAI_API_KEY"] = ""
os.environ["GOOGLE_API_KEY"] = ""

In [ ]:
from openai import OpenAI
gpt = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

In [ ]:
import google.generativeai as genai
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
# List available models to find a supported one
# for m in genai.list_models():
#   if 'generateContent' in m.supported_generation_methods:
#     print(m.name)
GEMINI_MODEL = "models/gemini-2.5-flash" # Update with a supported model if the above list reveals a different one.

In [ ]:
def chain_of_thought(problem_text):
    return f"""
You are a skilled Python developer.
{problem_text}
Think step by step about the logic before writing the code.
Then write the final, correct function implementation.
"""

def stepwise_chain_of_thought(problem_text):
    return f"""
You are to solve the following problem step-by-step.

1. Break down the logic clearly in small numbered steps.
2. Then write the Python function.
3. Double-check for correctness and edge cases.

Problem:
{problem_text}
"""

In [ ]:
def generate_code_gpt(prompt):
    response = gpt.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )
    return response.choices[0].message.content.strip()

def generate_code_gemini(prompt):
    model = genai.GenerativeModel(GEMINI_MODEL)
    response = model.generate_content(prompt)
    return response.text.strip()


In [ ]:
import io, contextlib

def check_pass(problem, code):
    full_code = code + "\n" + problem["test"]
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            exec(full_code, {})
        return 1
    except Exception as e:
        return 0


Part1

In [ ]:
import pandas as pd

# select 10 problems as dictionary
selected = {k: problems[k] for k in list(problems.keys())[:10]}
print("Selected problems:", list(selected.keys()))

strategies = [("CoT", chain_of_thought), ("SCoT", stepwise_chain_of_thought)]
records = []

for pid, prob in selected.items():
    for strat_name, strat_fn in strategies:
        prompt = strat_fn(prob["prompt"])

        # GPT
        gpt_code = generate_code_gpt(prompt)
        gpt_pass = check_pass(prob, gpt_code)
        records.append([pid, "GPT-4o-mini", strat_name, gpt_pass])

        # Gemini
        gem_code = generate_code_gemini(prompt)
        gem_pass = check_pass(prob, gem_code)
        records.append([pid, GEMINI_MODEL, strat_name, gem_pass])

df = pd.DataFrame(records, columns=["Problem", "Model", "PromptType", "Pass@1"])
df.to_csv("results_part1.csv", index=False)
df


Selected problems: ['HumanEval/0', 'HumanEval/1', 'HumanEval/2', 'HumanEval/3', 'HumanEval/4', 'HumanEval/5', 'HumanEval/6', 'HumanEval/7', 'HumanEval/8', 'HumanEval/9']


,Problem,Model,PromptType,Pass@1
0,HumanEval/0,GPT-4o-mini,CoT,0
1,HumanEval/0,gemini-2.5-flash,CoT,0
2,HumanEval/0,GPT-4o-mini,SCoT,0
3,HumanEval/0,gemini-2.5-flash,SCoT,0
4,HumanEval/1,GPT-4o-mini,CoT,0
5,HumanEval/1,gemini-2.5-flash,CoT,0
6,HumanEval/1,GPT-4o-mini,SCoT,0
7,HumanEval/1,gemini-2.5-flash,SCoT,0
8,HumanEval/2,GPT-4o-mini,CoT,0
9,HumanEval/2,gemini-2.5-flash,CoT,0


Pass@1 = 0 → the generated code failed at least one test case in the HumanEval problem.

Pass@1 = 1 → the generated code passed all test cases.

In [ ]:
import pandas as pd
import re
import os
from google.colab import files


# --- Step 1: Utility functions ---

def extract_code(text):
    """
    Remove markdown or explanation text to leave clean Python code.
    """
    if not text:
        return ""
    code = re.sub(r"```(?:python)?\n", "", text)
    code = re.sub(r"```", "", code)
    return code.strip()

def generate_code_gpt(prompt):
    """
    Replace with your GPT API call
    """
    # Example for OpenAI
    import openai
    response = openai.ChatCompletion.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    text = response.choices[0].message.content
    return extract_code(text)

def generate_code_gemini(prompt):
    """
    Uses Google Gemini
    """
    model = genai.GenerativeModel(GEMINI_MODEL)
    response = model.generate_content(prompt)
    if hasattr(response, "text") and response.text:
        return extract_code(response.text)
    elif hasattr(response, "candidates") and response.candidates:
        return extract_code(response.candidates[0].content.parts[0].text)
    else:
        return ""

def check_pass(problem, code):
    """
    Execute the problem's test cases safely
    Returns 1 if all pass, 0 if any fail
    """
    try:
        # 'problem["test"]' should contain the test code
        local_vars = {}
        exec(code, {}, local_vars)
        exec(problem["test"], {}, local_vars)
        return 1
    except Exception as e:
        return 0

# --- Step 2: Refined prompting functions ---

def chain_of_thought_refined(problem_text):
    return f"""
You are a Python expert.
Write a function that strictly follows the given signature.
Include all edge cases (empty lists, negative numbers, zero, invalid inputs if applicable).
Do not add any explanation — only provide the function code.

Problem:
{problem_text}
"""

def stepwise_chain_of_thought_refined(problem_text):
    return f"""
You are a Python expert.
Solve the problem step by step.
1. Plan your approach.
2. Implement the function following the signature.
3. Include edge cases.
Do not add explanations outside the code.

Problem:
{problem_text}
"""

# --- Step 3: Select 10 problems ---
selected = {k: problems[k] for k in list(problems.keys())[:10]}
print("Selected problems:", list(selected.keys()))

# --- Step 4: Run evaluation ---
strategies = [("CoT", chain_of_thought_refined), ("SCoT", stepwise_chain_of_thought_refined)]
records = []

for pid, prob in selected.items():
    for strat_name, strat_fn in strategies:
        prompt = strat_fn(prob["prompt"])

        # GPT
        gpt_code = generate_code_gpt(prompt)
        gpt_pass = check_pass(prob, gpt_code)
        records.append([pid, "GPT-4o-mini", strat_name, gpt_pass, prompt, gpt_code])

        # Gemini
        gem_code = generate_code_gemini(prompt)
        gem_pass = check_pass(prob, gem_code)
        records.append([pid, GEMINI_MODEL, strat_name, gem_pass, prompt, gem_code])

# --- Step 5: Save results ---
df = pd.DataFrame(records, columns=["Problem", "Model", "PromptType", "Pass@1", "Prompt", "GeneratedCode"])
csv_path = "/content/results_part1_refined.csv"
df.to_csv(csv_path, index=False)
files.download(csv_path)
df


Selected problems: ['HumanEval/0', 'HumanEval/1', 'HumanEval/2', 'HumanEval/3', 'HumanEval/4', 'HumanEval/5', 'HumanEval/6', 'HumanEval/7', 'HumanEval/8', 'HumanEval/9']
False
False
True
False
False
['()', '(())', '(()())']
['()']
['((()))']
['()', '()', '()']
['()', '(())', '(())']
[]
False
True
False
True
True
0.0
0.0
0.0
0.6666666666666666
[]
[1, 4, 2, 4, 3]
[5]
[1, -1, 2]
[1, 0, 2, 0, 3, 0, 4]
[2, 3, 1, 3]
[1, 1, 1]
[]
[5]
[1]
(0, 1)
(10, 24)
(-6, -6)
(3, 0)
(5, 5)
[]
[5]
[-1, -1, -1, -1]
[1, 1, 1, 1]
[1, 3, 3, 5, 5]


,Problem,Model,PromptType,Pass@1,Prompt,GeneratedCode
0,HumanEval/0,GPT-4o-mini,CoT,1,\nYou are a Python expert.\nWrite a function t...,from typing import List\n\ndef has_close_eleme...
1,HumanEval/0,gemini-2.5-flash,CoT,1,\nYou are a Python expert.\nWrite a function t...,from typing import List\n\n\ndef has_close_ele...
2,HumanEval/0,GPT-4o-mini,SCoT,1,\nYou are a Python expert.\nSolve the problem ...,from typing import List\n\ndef has_close_eleme...
3,HumanEval/0,gemini-2.5-flash,SCoT,1,\nYou are a Python expert.\nSolve the problem ...,from typing import List\n\n\ndef has_close_ele...
4,HumanEval/1,GPT-4o-mini,CoT,1,\nYou are a Python expert.\nWrite a function t...,from typing import List\n\ndef separate_paren_...
5,HumanEval/1,gemini-2.5-flash,CoT,1,\nYou are a Python expert.\nWrite a function t...,from typing import List\n\n\ndef separate_pare...
6,HumanEval/1,GPT-4o-mini,SCoT,1,\nYou are a Python expert.\nSolve the problem ...,from typing import List\n\ndef separate_paren_...
7,HumanEval/1,gemini-2.5-flash,SCoT,1,\nYou are a Python expert.\nSolve the problem ...,from typing import List\n\n\ndef separate_pare...
8,HumanEval/2,GPT-4o-mini,CoT,1,\nYou are a Python expert.\nWrite a function t...,def truncate_number(number: float) -> float:\n...
9,HumanEval/2,gemini-2.5-flash,CoT,1,\nYou are a Python expert.\nWrite a function t...,import math\n\ndef truncate_number(number: flo...


In [ ]:
!pip install datasets huggingface_hub gdown

In [ ]:
!huggingface-cli login

⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.

    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) y
Token is valid (permission: fineGrained).
The token `520ex1` has been saved to /root/.cache/huggingface/stored_tokens
Cannot authenticate through git-creden

In [ ]:
# ------------------------
# Load APPS Dataset (Official)
# ------------------------
from datasets import load_dataset

# Choose "default" or "verified"
apps_dataset = load_dataset("Elfsong/APPS", "default", split="test")

# Filter only harder problems (competition-level)
apps_hard = [p for p in apps_dataset if p.get("difficulty", "") == "competition"]
selected_apps = apps_hard[:3]  # select 5 challenging problems

print(f"Loaded {len(selected_apps)} hard APPS problems for Part 2.\n")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/41.0M [00:00<?, ?B/s]

data/test-00000-of-00003.parquet:   0%|          | 0.00/256M [00:00<?, ?B/s]

data/test-00001-of-00003.parquet:   0%|          | 0.00/184M [00:00<?, ?B/s]

data/test-00002-of-00003.parquet:   0%|          | 0.00/303M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4805 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3765 [00:00<?, ? examples/s]

Loaded 3 hard APPS problems for Part 2.



In [ ]:
selected_apps[1]

{'problem_id': 3690,
 'question': 'Have you ever tried to explain to the coordinator, why it is eight hours to the contest and not a single problem has been prepared yet? Misha had. And this time he has a really strong excuse: he faced a space-time paradox! Space and time replaced each other.\n\nThe entire universe turned into an enormous clock face with three hands\xa0— hour, minute, and second. Time froze, and clocks now show the time h hours, m minutes, s seconds.\n\nLast time Misha talked with the coordinator at t_1 o\'clock, so now he stands on the number t_1 on the clock face. The contest should be ready by t_2 o\'clock. In the terms of paradox it means that Misha has to go to number t_2 somehow. Note that he doesn\'t have to move forward only: in these circumstances time has no direction.\n\nClock hands are very long, and Misha cannot get round them. He also cannot step over as it leads to the collapse of space-time. That is, if hour clock points 12 and Misha stands at 11 then h

In [ ]:
print(apps_dataset[0])

{'problem_id': 0, 'question': "An accordion is a string (yes, in the real world accordions are musical instruments, but let's forget about it for a while) which can be represented as a concatenation of: an opening bracket (ASCII code $091$), a colon (ASCII code $058$), some (possibly zero) vertical line characters (ASCII code $124$), another colon, and a closing bracket (ASCII code $093$). The length of the accordion is the number of characters in it.\n\nFor example, [::], [:||:] and [:|||:] are accordions having length $4$, $6$ and $7$. (:|:), {:||:}, [:], ]:||:[ are not accordions. \n\nYou are given a string $s$. You want to transform it into an accordion by removing some (possibly zero) characters from it. Note that you may not insert new characters or reorder existing ones. Is it possible to obtain an accordion by removing characters from $s$, and if so, what is the maximum possible length of the result?\n\n\n-----Input-----\n\nThe only line contains one string $s$ ($1 \\le |s| \\l

In [ ]:
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

In [ ]:
# ===========================================
# PART 2: Introduce Harder Problems from APPS
# ===========================================

!pip install -q datasets google-generativeai openai pandas tqdm

import os
import re
import pandas as pd
from datasets import load_dataset
from tqdm import tqdm
import google.generativeai as genai
from openai import OpenAI

# ------------------------
# Environment setup
# ------------------------
os.environ["OPENAI_API_KEY"] = ""
os.environ["GOOGLE_API_KEY"] = ""

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
GEMINI_MODEL = "gemini-2.5-flash"

# ------------------------
# Utility Functions
# ------------------------
def extract_code(text):
    if not text:
        return ""
    code = re.sub(r"```(?:python)?\n", "", text)
    code = re.sub(r"```", "", code)
    return code.strip()

def generate_code_gpt(prompt, temperature=0.7):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
    )
    return extract_code(response.choices[0].message.content)

def generate_code_gemini(prompt, temperature=0.7):
    model = genai.GenerativeModel(GEMINI_MODEL)
    try:
        response = model.generate_content(
            prompt,
            generation_config={"temperature": temperature},
        )
        if hasattr(response, "text") and response.text:
            return extract_code(response.text)
        elif hasattr(response, "candidates") and response.candidates:
            return extract_code(response.candidates[0].content.parts[0].text)
        else:
            return ""
    except Exception as e:
        return f"# Gemini Error: {e}"

def check_pass(problem, code):
    try:
        local_vars = {}
        exec(code, {}, local_vars)
        exec(problem["test"], {}, local_vars)
        return 1, ""
    except Exception as e:
        return 0, str(e)

# ------------------------
# Prompt Templates
# ------------------------
def chain_of_thought_refined(problem_text):
    return f"""
You are a Python expert.
Write a function that strictly follows the given signature.
Include all edge cases (empty lists, invalid inputs, performance limits).
Do not add any explanation — only provide the function code.

Problem:
{problem_text}
"""

def stepwise_chain_of_thought_refined(problem_text):
    return f"""
You are a Python expert.
Solve the problem step by step.
1. Plan your approach.
2. Implement the function following the signature.
3. Handle edge cases and constraints.
Do not add explanations outside the code.

Problem:
{problem_text}
"""

# # ------------------------
# # Load APPS Dataset (New Source)
# # ------------------------
# # "dz1/CodeScore-APPS" is a modern, script-free APPS dataset mirror
# apps_dataset = load_dataset("dz1/CodeScore-APPS", split="test")

# Filter only harder problems (competition-level)
apps_hard = [p for p in apps_dataset if p.get("difficulty", "") == "competition"]
selected_apps = apps_hard[:3]  # select 5 challenging problems

# print(f"Loaded {len(selected_apps)} hard APPS problems for Part 2.\n")

# ------------------------
# Evaluation Loop
# ------------------------
strategies = [
    ("CoT", chain_of_thought_refined),
    ("SCoT", stepwise_chain_of_thought_refined)
]

records = []

for idx, prob in enumerate(tqdm(selected_apps, desc="Evaluating APPS Hard Problems")):
    prob_text = prob.get("question", "")
    test_code = prob.get("input_output", "")

    problem = {"prompt": prob_text, "test": test_code}

    for strat_name, strat_fn in strategies:
        prompt = strat_fn(problem["prompt"])

        # GPT
        gpt_code = generate_code_gpt(prompt)
        gpt_pass, gpt_err = check_pass(problem, gpt_code)
        records.append([idx, "GPT-4o-mini", strat_name, gpt_pass, prompt, gpt_code, gpt_err])

        # Gemini
        gem_code = generate_code_gemini(prompt)
        gem_pass, gem_err = check_pass(problem, gem_code)
        records.append([idx, GEMINI_MODEL, strat_name, gem_pass, prompt, gem_code, gem_err])

# ------------------------
# Save Results
# ------------------------
df = pd.DataFrame(records, columns=[
    "ProblemID", "Model", "PromptType", "Pass@1", "Prompt", "GeneratedCode", "Error"
])
df.to_csv("apps_failures_part2.csv", index=False)
print("\n✅ Results saved to apps_failures_part2.csv")
df.head()
df

Evaluating APPS Hard Problems:   0%|          | 0/3 [00:00<?, ?it/s]

4
1313


Evaluating APPS Hard Problems:   0%|          | 0/3 [25:36<?, ?it/s]


KeyboardInterrupt: Interrupted by user

In [ ]:
prompt = "Write a Python function to check if a number is prime."
print("🔹 Testing Gemini...")
try:
    from google.generativeai import GenerativeModel
    model = GenerativeModel("gemini-1.5-flash")
    response = model.generate_content(prompt)
    print("✅ Gemini response:", response.text[:200])
except Exception as e:
    print("❌ Gemini error:", e)


🔹 Testing Gemini...


❌ Gemini error: 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.


In [ ]:
import google.generativeai as genai

genai.configure(api_key="")

for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(m.name)


models/gemini-2.5-pro-preview-03-25
models/gemini-2.5-flash-preview-05-20
models/gemini-2.5-flash
models/gemini-2.5-flash-lite-preview-06-17
models/gemini-2.5-pro-preview-05-06
models/gemini-2.5-pro-preview-06-05
models/gemini-2.5-pro
models/gemini-2.0-flash-exp
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.0-flash-preview-image-generation
models/gemini-2.0-flash-lite-preview-02-05
models/gemini-2.0-flash-lite-preview
models/gemini-2.0-pro-exp
models/gemini-2.0-pro-exp-02-05
models/gemini-exp-1206
models/gemini-2.0-flash-thinking-exp-01-21
models/gemini-2.0-flash-thinking-exp
models/gemini-2.0-flash-thinking-exp-1219
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/learnlm-2.0-flash-experimental
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = ""

print("✅ Updated key:", os.environ["OPENAI_API_KEY"][:10], "...")  # just to verify it's set


✅ Updated key: sk-proj-3A ...


In [ ]:
from openai import OpenAI
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

try:
    models = client.models.list()
    print("✅ OpenAI connection successful! Total models:", len(models.data))
except Exception as e:
    print("❌ Connection failed:", e)


✅ OpenAI connection successful! Total models: 76
